In [1]:
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, CosineAnnealingLR

from typing import Dict, List, Tuple, Optional, Any, Type, Union

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
import copy

import matplotlib.pyplot as plt
import seaborn as sns

import import_ipynb
import os
curdir = os.getcwd()
src_path = Path('../src/')
os.chdir(src_path)
from ML_Base import XANESVisualizer
os.chdir(curdir)

## Необходимые для блокнота функции

In [2]:
def fit_cyclic_checkpoints(model_cls: Type[nn.Module],
                           model_params: Dict[str, Any],
                           criterion: nn.Module,
                           X_train: np.ndarray,
                           y_train: np.ndarray,
                           X_val_split: np.ndarray,
                           y_val_split: np.ndarray,
                           X_val_indep: np.ndarray,
                           y_val_indep: np.ndarray,
                           optimizer_cls: Type[torch.optim.Optimizer],
                           optimizer_params: Dict[str, Any],
                           n_targets: Optional[int] = None,
                           target_index: Optional[int] = None,
                           scheduler_cls: Optional[Type[Any]] = None,
                           scheduler_params: Optional[Dict[str, Any]] = None,
                           X_test: Optional[np.ndarray] = None,
                           y_test: Optional[np.ndarray] = None,
                           batch_size: int = 128,
                           num_epochs: int = 500,
                           save_dir: str = "./checkpoints",
                           device: Optional[torch.device] = None
                          ) -> Tuple[nn.Module, Dict[str, List[float]], Dict[str, Any]]:
    '''
    Заточено под использование с CosineAnnealingWarmRestarts.
    Определяет границы циклов обучения по следующим параметрам:
        T_0 = scheduler_params.get('T_0', 80)
        T_mult = scheduler_params.get('T_mult', 1)
    Возвращает model, history, artifacts

    n_targets - опционально, указывает сколько первых коорд. чисел рассчитывать, если None то считает для всех доступных 7 сфер.
    target_index - замена n_targets, указывает какое конкретно КЧ брать в качестве таргета.
    '''
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
    os.makedirs(save_dir, exist_ok=True)
        
    # Подготовка данных
    if target_index is None:
        n_targets = 7 if n_targets is None else n_targets
        y_train_t = torch.FloatTensor(y_train[:, :n_targets])
        y_val_split_t = torch.FloatTensor(y_val_split[:, :n_targets])
        y_val_indep_t = torch.FloatTensor(y_val_indep[:, :n_targets])
    else:
        y_train_t = torch.FloatTensor(y_train[:, target_index]).view(-1, 1)
        y_val_split_t = torch.FloatTensor(y_val_split[:, target_index]).view(-1, 1)
        y_val_indep_t = torch.FloatTensor(y_val_indep[:, target_index]).view(-1, 1)
    
    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), y_train_t), batch_size=batch_size, shuffle=True)
    val_split_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val_split), y_val_split_t), batch_size=batch_size, shuffle=False)
    val_indep_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val_indep), y_val_indep_t), batch_size=batch_size, shuffle=False)
    
    test_loader = None
    if X_test is not None and y_test is not None:
        if target_index is None:
            y_test_t = torch.FloatTensor(y_test[:, :n_targets])
        else:
            y_test_t = torch.FloatTensor(y_test[:, target_index]).view(-1, 1)
        test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test), y_test_t), batch_size=batch_size, shuffle=False)

    # Инициализация модели, оптимизатора, шедулера
    model = model_cls(num_outputs=n_targets, **model_params).to(device)
    optimizer = optimizer_cls(model.parameters(), **optimizer_params)
    
    scheduler = None
    if scheduler_cls and scheduler_params:
        scheduler = scheduler_cls(optimizer, **scheduler_params)
        
    # Вычисляем границы циклов отжига на основе T_0 и T_mult
    T_0 = scheduler_params.get('T_0', 80)
    T_mult = scheduler_params.get('T_mult', 1)
    
    cycle_boundaries = [0]
    curr_T = T_0
    while cycle_boundaries[-1] < num_epochs:
        cycle_boundaries.append(cycle_boundaries[-1] + curr_T)
        curr_T *= T_mult
        
    def get_active_cycle(epoch_idx):
        for cid, (start, end) in enumerate(zip(cycle_boundaries[:-1], cycle_boundaries[1:])):
            if start <= epoch_idx < end:
                return cid
        return len(cycle_boundaries) - 2

    history = {
        'train_loss': [], 'train_mae': [], 'train_mse': [], 'train_r2': [], 'train_std': [],
        'val_indep_loss': [], 'val_indep_mae': [], 'val_indep_mse': [], 'val_indep_r2': [], 'val_indep_std': [],
        'val_split_loss': [], 'val_split_mae': [], 'val_split_mse': [], 'val_split_r2': [], 'val_split_std': []
    }
    if test_loader is not None:
        history['test_loss'] = []
        history['test_mae'] = []
        history['test_mse'] = []
        history['test_r2'] = []
        history['test_std'] = []

    # Реестр для хранения лучших моделей по каждому циклу
    cycle_best_checkpoints = {}
    current_cycle_best = {'loss': float('inf'), 
                          'ckpt_info': {'epoch': None,
                                        'val_indep_r2': float('-inf'),
                                        'val_indep_mae': float('inf'),
                                        'val_indep_mse': float('inf'),
                                        'val_indep_std': float('inf'),
                                       }
                         }
    
    active_cycle = 0
    
    # Фильтр по R² (например, модель должна давать R² хотя бы 0.80)
    MIN_R2_THRESHOLD = 0.80

    for epoch in range(num_epochs):
        new_cycle = get_active_cycle(epoch)
        if new_cycle != active_cycle:
            c_id = active_cycle
            
            if current_cycle_best['ckpt_info']['epoch'] is not None:
                best_info = current_cycle_best['ckpt_info']
                path = os.path.join(save_dir, 
                                    f"cycle_{c_id}_epoch_{best_info['epoch']}_loss_{current_cycle_best['loss']:.3f}.pth"
                                   )
                torch.save(best_info, path)
                cycle_best_checkpoints[c_id] = {'path': path, **best_info}
                
                print(f"🔄 [Цикл {c_id} завершен] Сохранен лучший чекпоинт: "
                      f"эпоха {best_info['epoch']} | "
                      f"Loss: {current_cycle_best['loss']:.3f} | "
                      f"R²: {best_info['val_indep_r2']:.3f} | "
                      f"MAE: {best_info['val_indep_mae']:.3f} "
                     )
            else:
                print(f"⚠️ [Цикл {c_id} завершен] Ни одна модель не преодолела порог R² >= {MIN_R2_THRESHOLD}. Чекпоинт не сохранен.")
            
            active_cycle = new_cycle
            current_cycle_best = {'loss': float('inf'), 
                                  'ckpt_info': {'epoch': None,
                                                'val_indep_r2': float('-inf'),
                                                'val_indep_mae': float('inf'),
                                                'val_indep_mse': float('inf'),
                                                'val_indep_std': float('inf'),
                                               }
                                 }

        # Тренировка модели
        model.train()
        running_train_loss = 0.0
        total_train_samples = 0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            b_size = inputs.size(0)
            running_train_loss += loss.item() * b_size
            total_train_samples += b_size
            
        # Шаг шедулера на каждую эпохУ
        if scheduler is not None:
            scheduler.step()
            
        epoch_train_loss = running_train_loss / total_train_samples
        history['train_loss'].append(epoch_train_loss)
        
        # Сбор метрик
        _, train_metrics = evaluate_metrics(model, train_loader, criterion, device)
        val_split_loss, val_split_metrics = evaluate_metrics(model, val_split_loader, criterion, device)
        val_indep_loss, val_indep_metrics = evaluate_metrics(model, val_indep_loader, criterion, device)
        if test_loader is not None:
            test_loss, test_metrics = evaluate_metrics(model, test_loader, criterion, device)
        
        # Запись в историю
        history["val_indep_loss"].append(val_indep_loss)
        history["val_split_loss"].append(val_split_loss)
        if test_loader is not None:
            history["test_loss"].append(test_loss)
        
        for m_name in ['mae', 'mse', 'r2', 'std']:
            history[f"train_{m_name}"].append(train_metrics[m_name])
            history[f"val_indep_{m_name}"].append(val_indep_metrics[m_name])
            history[f"val_split_{m_name}"].append(val_split_metrics[m_name])
            if test_loader is not None:
                history[f"test_{m_name}"].append(test_metrics[m_name])

        val_mae = val_indep_metrics['mae']
        val_mse = val_indep_metrics['mse']
        val_r2 = val_indep_metrics['r2']
        val_std = val_indep_metrics['std']
        
        prev_mae = current_cycle_best['ckpt_info']['val_indep_mae']
        prev_std = current_cycle_best['ckpt_info']['val_indep_std']
        prev_r2 = current_cycle_best['ckpt_info']['val_indep_r2']
        
        # Задаем минимальный порог значимого улучшения (например, 0.5% для MAE и STD)
        TOLERANCE_MAE = 0.005  # 0.5%
        TOLERANCE_STD = 0.005  # 0.5%
        
        if val_r2 >= MIN_R2_THRESHOLD:
            r2_ok = val_r2 >= (prev_r2 - 0.01)
            
            if r2_ok:

                mae_improved = val_mae < prev_mae * (1.0 - TOLERANCE_MAE)
                std_within_tolerance = val_std <= prev_std * 1.10  # люфт для второстепенной метрики
                
                std_improved = val_std < prev_std * (1.0 - TOLERANCE_STD)
                mae_within_tolerance = val_mae <= prev_mae * 1.02  # люфт для второстепенной метрики
                
                if (mae_improved and std_within_tolerance) or (std_improved and mae_within_tolerance):
                    current_cycle_best['loss'] = val_indep_loss
                    current_cycle_best['ckpt_info'] = {'epoch': epoch + 1,
                                                       'cycle': active_cycle,
                                                       'model_state_dict': copy.deepcopy(model.state_dict()),
                                                       'optimizer_state_dict': copy.deepcopy(optimizer.state_dict()),
                                                       'val_indep_loss': val_indep_loss,
                                                       'val_indep_r2': val_r2,
                                                       'val_indep_mae': val_mae,
                                                       'val_indep_mse': val_mse,
                                                       'val_indep_std': val_std,
                                                      }

        # Логгирование
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] (Cycle {active_cycle}) | "
                  f"Train: {epoch_train_loss:.3f} | "
                  f"ValIndep Loss: {val_indep_loss:.3f} | "
                  f"ValIndep R2: {val_indep_metrics['r2']:.3f} | "
                  f"ValIndep MAE: {val_indep_metrics['mae']:.3f} "
                  f"ValIndep STD: {val_indep_metrics['std']:.3f}")

    # Сохраняем финалиста последнего активного цикла
    if current_cycle_best['ckpt_info']['epoch'] is not None:
        c_id = active_cycle
        path = os.path.join(save_dir, f"cycle_{c_id}_epoch_{current_cycle_best['ckpt_info']['epoch']}_loss_{current_cycle_best['loss']:.3f}.pth")
        torch.save(current_cycle_best['ckpt_info'], path)
        cycle_best_checkpoints[c_id] = {'path': path, **current_cycle_best['ckpt_info']}

    # Находим лучшую модель по всем циклам
    if not cycle_best_checkpoints:
        raise RuntimeError(f"❌ Ошибка: ни в одном цикле не нашлось моделей, прошедших фильтрацию (R² >= {MIN_R2_THRESHOLD}).")
    best_overall_cycle = min(cycle_best_checkpoints.values(), key=lambda x: x['val_indep_mae'])
    model.load_state_dict(best_overall_cycle['model_state_dict'])
    print(f"\n🏆 Загружена лучшая модель из Цикла {best_overall_cycle['cycle']}: эпоха {best_overall_cycle['epoch']} (Val Indep Loss: {best_overall_cycle['val_indep_loss']:.3f}, R²: {best_overall_cycle['val_indep_r2']:.3f})")

    artifacts = {"optimizer": optimizer,
                 "scheduler": scheduler,
                 "cycle_checkpoints": cycle_best_checkpoints
                }

    return model, history, artifacts


def evaluate_metrics(model: nn.Module, data_loader: DataLoader, criterion: nn.Module, device: torch.device) -> Tuple[float, Dict[str, float]]:
    '''
    Считает loss и развернутые физические метрики (MAE, MSE, R2, STD)
    '''
    model.eval()
    running_loss = 0.0
    total_samples = 0
    
    all_mu = []
    all_targets = []
    all_std = []
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs) # (mu, log_var)
            loss = criterion(outputs, targets)
            
            mu, log_var = outputs
            std = torch.sqrt(torch.exp(log_var))
            
            b_size = inputs.size(0)
            running_loss += loss.item() * b_size
            total_samples += b_size
            
            all_mu.append(mu.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            all_std.append(std.cpu().numpy())
            
    epoch_loss = running_loss / total_samples
    
    mu_arr = np.concatenate(all_mu, axis=0)
    target_arr = np.concatenate(all_targets, axis=0)
    std_arr = np.concatenate(all_std, axis=0)
    
    metrics = {'mae': mean_absolute_error(target_arr, mu_arr),
               'mse': mean_squared_error(target_arr, mu_arr),
               'r2': r2_score(target_arr, mu_arr),
               'std': float(std_arr.mean())
              }
    
    return epoch_loss, metrics

In [3]:
def get_probabilistic_feature_importance(model, X_sample, device, target_idx=0, output_type='mu'):
    '''
    Saliency Map для вероятностной модели: показывает важность точек спектра 
    отдельно для предсказания значения (mu) или неопределенности (std/log_var) 
    для конкретного координационного числа (target_idx).
    
    Parameters:
    - target_idx: индекс целевого КЧ (от 0 to num_outputs-1). 
    - output_type: 'mu' (для среднего) или 'std' (для стандартного отклонения).
    '''
    model.eval()
    
    x = torch.FloatTensor(X_sample).to(device)
    if x.ndim == 1:
        x = x.unsqueeze(0)
        
    x.requires_grad_(True)
    
    # Прямой проход: получаем оба выхода модели
    mu, log_var = model(x)
    
    if output_type == 'mu':
        # Берем предсказание mu для конкретного выхода
        target_output = mu[:, target_idx]
    elif output_type == 'std':
        # Превращаем log_var в std (σ = sqrt(exp(log_var))) и берем конкретный выход
        std = torch.sqrt(torch.exp(log_var))
        target_output = std[:, target_idx]
    else:
        raise ValueError("output_type должен быть 'mu' или 'std'")
    
    # Считаем градиент только по выбранному скалярному выходу
    # Если на входе батч, суммируем по батчу, чтобы получить общий градиент
    scalar_output = target_output.sum()
    scalar_output.backward()
    
    # Извлекаем градиент, усредняем по батчу (если батч > 1) и берем модуль
    importance = x.grad.abs().mean(dim=0).cpu().numpy()
    
    return importance

In [4]:
class ProbabilisticRegressionHead(nn.Module):
    '''
    Голова модели: выдает линейные предсказания для Mu и Log(Variance) 
    для каждого из целевых параметров (num_outputs от 1 до 7).
    '''
    def __init__(self, in_features: int, num_outputs: int):
        super().__init__()
        self.fc_mu = nn.Linear(in_features, num_outputs)
        self.fc_log_var = nn.Linear(in_features, num_outputs)
        # self.sigmoid = nn.Sigmoid()
        
        # Инициализируем голову дисперсии так, чтобы на старте 
        # модель была уверена в прогнозе (sigma ≈ 0.22 -> log_var ≈ -3.0)
        nn.init.constant_(self.fc_log_var.weight, 1e-6)
        nn.init.constant_(self.fc_log_var.bias, -3.0)

    def forward(self, x: torch.Tensor):
        mu = self.fc_mu(x)
        # mu = self.sigmoid(mu)
        # log_var = torch.zeros_like(mu)
        log_var = self.fc_log_var(x)
        log_var = torch.clamp(log_var, min=-10.0, max=1.5)
        return mu, log_var

class GaussianNLLLoss(nn.Module):
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, outputs, target: torch.Tensor) -> torch.Tensor:
        if isinstance(outputs, (tuple, list)):
            mu, log_var = outputs
        else:
            mu, log_var = torch.chunk(outputs, chunks=2, dim=-1)
            
        var = torch.exp(log_var) + self.eps
        loss = 0.5 * log_var + 0.5 * ((target - mu) ** 2) / var
        return loss.mean()

class HybridGaussianMSELoss(nn.Module):
    def __init__(self, mse_weight=1.0, mae_weight=1.0, nll_weight=1.0, eps=1e-6):
        super().__init__()
        self.mse_weight = mse_weight
        self.mae_weight = mae_weight
        self.nll_weight = nll_weight
        self.eps = eps
        self.mae = nn.L1Loss()
        self.mse = nn.MSELoss()
        self.nnl_loss = GaussianNLLLoss()

    def forward(self, outputs, target: torch.Tensor) -> torch.Tensor:
        if isinstance(outputs, (tuple, list)):
            mu, log_var = outputs
        else:
            mu, log_var = torch.chunk(outputs, chunks=2, dim=-1)
            
        var = torch.exp(log_var) + self.eps
        nll_loss = self.nnl_loss(outputs, target)
        
        mse_loss = self.mse(mu, target)
        mae_loss = self.mae(mu, target)
        
        # Итоговая взвешенная сумма
        total_loss = self.nll_weight * nll_loss + self.mse_weight * mse_loss + self.mae_weight * mae_loss
        return total_loss

In [5]:
class XANES_CNN_Stable(nn.Module):
    def __init__(self, in_features, num_outputs=4, kernel_size=3, dropout=0.0):
        super().__init__()
        
        # Блок 1
        self.conv1 = nn.Conv1d(1, 16, kernel_size=kernel_size, padding='same')
        self.bn1 = nn.BatchNorm1d(16)
        
        # Блок 2
        self.conv2 = nn.Conv1d(16, 32, kernel_size=kernel_size, padding='same')
        self.bn2 = nn.BatchNorm1d(32)
        
        # Блок 3 с остаточной связью (residual connection)
        self.conv3 = nn.Conv1d(32, 64, kernel_size=kernel_size, padding='same')
        self.bn3 = nn.BatchNorm1d(64)
        # Шорткат для согласования каналов (32 -> 64) на третьем блоке
        self.shortcut3 = nn.Conv1d(32, 64, kernel_size=1) 
        
        self.pool = nn.MaxPool1d(2)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        
        self.global_pool = nn.AdaptiveAvgPool1d(10)
        
        self.fc = nn.Sequential(
            nn.Linear(64 * 10, 64),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.prob_head = ProbabilisticRegressionHead(in_features=64, num_outputs=num_outputs)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
            
        # Блок 1
        out = self.act(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.pool(out)
        
        # Блок 2
        out = self.act(self.bn2(self.conv2(out)))
        out = self.dropout(out)
        out = self.pool(out)
        
        # Блок 3 с Residual Connection (дает стабильность градиентов)
        identity = self.shortcut3(out)
        out = self.conv3(out)
        out = self.bn3(out)
        out = self.act(out + identity)
        out = self.dropout(out)
        
        out = self.global_pool(out)
        out = out.flatten(1)
        feat = self.fc(out)
        
        mu, log_var = self.prob_head(feat)
        
        return mu, log_var

In [6]:
def plot_metrics(history: dict, save_path: Union[Path, str]):
    # Определяем количество эпох
    first_key = list(history.keys())[0]
    epochs = range(1, len(history[first_key]) + 1)

    fig = plt.figure(figsize=(14, 12))
    gs = plt.GridSpec(3, 3, figure=fig)
    
    ax_loss = fig.add_subplot(gs[0, :])  # Строка 0, во всю ширину
    ax_mae = fig.add_subplot(gs[1, 0]) # Строка 1, левая колонка
    ax_std = fig.add_subplot(gs[1, 1]) # Строка 1, средняя колонка
    ax_mse = fig.add_subplot(gs[1, 2])     # Строка 1 правая колонка
    ax_r2 = fig.add_subplot(gs[2, :])    # Строка 2, во всю ширину
    
    split_styles = {'train': {
                             'color': '#d62728',  # Красный
                             'marker': 'o',
                             'label_prefix': 'Train'
                            },
                    'val_split': {
                            'color': '#ff7f0e',  # Оранжевый
                            'marker': 's',
                            'label_prefix': 'Val (Split)'
                                },
                    'val_indep': {
                                'color': '#1f77b4',  # Синий
                                'marker': '^',
                                'label_prefix': 'Val (Indep)'
                                },
                    'test': {
                             'color': '#008080',  # Тёмно зеленый
                             'marker': 'd',
                             'label_prefix': 'Test'
                            },
                   }
    

    for split_key, style in split_styles.items():
        color = style['color']
        marker = style['marker']
        prefix = style['label_prefix']
        
        # 0 Loss
        loss_key = f"{split_key}_loss"
        if loss_key in history:
            ax_loss.plot(epochs, history[loss_key], label=f"{prefix} Loss", 
                         color=color, marker=marker, linewidth=2, alpha=0.8, markevery=max(1, len(epochs)//15))
        
        # 1-0 MAE
        mae_key = f"{split_key}_mae"
        if mae_key in history:
            ax_mae.plot(epochs, history[mae_key], label=f"{prefix} MAE", 
                        color=color, linestyle='-', marker=marker, linewidth=2, alpha=0.8, markevery=max(1, len(epochs)//15))
            
        # 1-1 STD
        std_key = f"{split_key}_std"
        if std_key in history:
            ax_std.plot(epochs, history[std_key], label=f"{prefix} STD", 
                        color=color, linestyle='-', marker=marker, linewidth=2, alpha=0.8, markevery=max(1, len(epochs)//15))

        # 1-2 MSE
        mse_key = f"{split_key}_mse"
        if mse_key in history:
            ax_mse.plot(epochs, history[mse_key], label=f"{prefix} MSE", 
                        color=color, linestyle='-', marker=marker, linewidth=2, alpha=0.8, markevery=max(1, len(epochs)//15))
            
        # 2 R2
        r2_key = f"{split_key}_r2"
        if r2_key in history:
            ax_r2.plot(epochs, history[r2_key], label=f"{prefix} R²", 
                       color=color, linestyle='-', marker=marker, linewidth=2, alpha=0.8, markevery=max(1, len(epochs)//15))

    # Оформление сабплотов 
    
    # Loss
    ax_loss.set_title('Функция потерь (Loss) по эпохам', fontsize=12, fontweight='bold')
    ax_loss.set_ylabel('Loss', fontsize=10)
    ax_loss.grid(True, linestyle='--', alpha=0.6)
    ax_loss.legend(loc='upper right')
    
    # MAE (отдельный график)
    ax_mae.set_title('Абсолютная ошибка (MAE)', fontsize=12, fontweight='bold')
    ax_mae.set_ylabel('MAE', fontsize=10)
    ax_mae.grid(True, linestyle='--', alpha=0.6)
    ax_mae.legend(loc='upper right')

    # Std (отдельный график)
    ax_std.set_title('Неопределенность (Std)', fontsize=12, fontweight='bold')
    ax_std.set_ylabel('Std', fontsize=10)
    ax_std.grid(True, linestyle='--', alpha=0.6)
    ax_std.legend(loc='upper right')
    
    
    # MSE (отдельный график)
    ax_mse.set_title('Среднеквадратичная ошибка (MSE)', fontsize=12, fontweight='bold')
    ax_mse.set_ylabel('MSE', fontsize=10)
    ax_mse.grid(True, linestyle='--', alpha=0.6)
    ax_mse.legend(loc='upper right')
    
    # R2
    ax_r2.set_title('Коэффициент детерминации (R² Score)', fontsize=12, fontweight='bold')
    ax_r2.set_xlabel('Эпохи', fontsize=10)
    ax_r2.set_ylabel('R²', fontsize=10)
    ax_r2.grid(True, linestyle='--', alpha=0.6)
    ax_r2.legend(loc='lower right')
    
    plt.tight_layout()
    plt.savefig(Path(save_path,'history.png', dpi=600))
    plt.show()

In [7]:
def evaluate_and_plot_board(model: nn.Module, 
                            device: torch.device,
                            n_out: int, 
                            X_train: np.ndarray,
                            y_train: np.ndarray,
                            X_test: np.ndarray,
                            y_test: np.ndarray,
                            X_val: Optional[np.ndarray] = None,
                            y_val: Optional[np.ndarray] = None,
                            X_indep: Optional[np.ndarray] = None,
                            y_indep: Optional[np.ndarray] = None,
                            target_idx: Optional[int] = None,
                            save_path: Optional[Union[str, Path]] = None
                           ):
    '''
    Диагностическая панель (4 строки × 3 столбца):
    - Строки: Train, Test, val_split (опц.), val_indep (опц.)
    - Столбцы: Parity Plot (с ±2σ), Residuals Distribution, Calibration (Pred STD vs Absolute Error).
    
    Если target_idx не указан, автоматически строит графики для каждого из n_out таргетов.
    '''
    model.eval()

    def get_predictions(X: np.ndarray, y: np.ndarray):
        """Вспомогательный инференс для получения mu, std и true значений"""
        if X is None or y is None:
            return None, None, None
        
        # Переводим в тензоры
        inputs = torch.tensor(X, dtype=torch.float32).to(device)
        targets = torch.tensor(y, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            outputs = model(inputs)
            mu, log_var = outputs
            mu = mu.cpu().numpy()
            log_var = log_var.cpu().numpy()
            std = np.exp(0.5 * log_var)
           
        return mu, std, targets.cpu().numpy()

    # Считаем предсказания для обязательных сплитов
    train_mu, train_std, train_true = get_predictions(X_train, y_train)
    test_mu, test_std, test_true = get_predictions(X_test, y_test)
    
    # Опциональные сплиты
    val_mu, val_std, val_true = get_predictions(X_val, y_val) if X_val is not None and y_val is not None else (None, None, None)
    indep_mu, indep_std, indep_true = get_predictions(X_indep, y_indep) if X_indep is not None and y_indep is not None else (None, None, None)

    # Определяем, какие таргеты обрабатывать
    targets_to_process = range(n_out) if target_idx is None else [target_idx]

    for t_idx in targets_to_process:
        cn_label = f"CN_{t_idx + 1}"
        
        # Собираем только те датасеты, которые реально переданы
        datasets_to_plot = [("Train Set", train_true[:, t_idx], train_mu[:, t_idx], train_std[:, t_idx], '#7f8c8d'),
                            ("Test Set", test_true[:, t_idx], test_mu[:, t_idx], test_std[:, t_idx], '#2c3e50')]
        if val_true is not None:
            datasets_to_plot.append(("Validation Split (val_split)", val_true[:, t_idx], val_mu[:, t_idx], val_std[:, t_idx], '#2980b9'))
        if indep_true is not None:
            datasets_to_plot.append(("Independent Validation (val_indep)", indep_true[:, t_idx], indep_mu[:, t_idx], indep_std[:, t_idx], '#16a085'))

        n_rows = len(datasets_to_plot)
        fig, axes = plt.subplots(n_rows, 3, figsize=(18, 4.5 * n_rows))
        
        if n_rows == 1:
            axes = np.array([axes])

        for row_idx, (title_prefix, t_arr, p_arr, s_arr, color) in enumerate(datasets_to_plot):
            row_axes = axes[row_idx]
            
            t_flat = t_arr.flatten()
            p_flat = p_arr.flatten()
            s_flat = s_arr.flatten()
            
            # --- Колонка 1: Parity Plot с доверительным интервалом ---
            row_axes[0].scatter(t_flat, p_flat, alpha=0.5, s=18, color=color, label='Predictions')
            row_axes[0].plot([t_flat.min(), t_flat.max()], [t_flat.min(), t_flat.max()], 'r--', lw=2)
            
            sort_idx = np.argsort(t_flat)
            row_axes[0].fill_between(
                t_flat[sort_idx], 
                (p_flat - 2*s_flat)[sort_idx], 
                (p_flat + 2*s_flat)[sort_idx], 
                color='#27ae60', alpha=0.2, label=f'Model ±2σ'
            )
            # row_axes[0].set_title(f'{cn_label} Parity — {title_prefix}')
            row_axes[0].set_title(f'Parity — {title_prefix}')
            row_axes[0].set_xlabel('True (Normalized)')
            row_axes[0].set_ylabel('Predicted')
            row_axes[0].legend(loc='upper left', fontsize=9)
            row_axes[0].grid(True, alpha=0.2)
            
            # --- Колонка 2: Residuals Distribution ---
            residuals = p_flat - t_flat
            sns.histplot(residuals, kde=True, ax=row_axes[1], color='#e74c3c', bins=30)
            row_axes[1].axvline(x=0, color='black', linestyle='--', lw=1.5)
            # row_axes[1].set_title(f'{cn_label} Residuals — {title_prefix}')
            row_axes[1].set_title(f'Residuals — {title_prefix}')
            row_axes[1].set_xlabel('Error (Pred - True)')
            row_axes[1].set_ylabel('Frequency')
            row_axes[1].grid(True, alpha=0.2)
            
            # --- Колонка 3: Calibration (Pred STD vs Absolute Error) ---
            abs_errors = np.abs(residuals)
            if len(abs_errors) > 1 and np.std(abs_errors) > 1e-8 and np.std(s_flat) > 1e-8:
                corr = np.corrcoef(abs_errors, s_flat)[0, 1]
            else:
                corr = 0.0
                
            row_axes[2].scatter(s_flat, abs_errors, alpha=0.4, s=18, color='#8e44ad')
            row_axes[2].set_title(f'Calibration ($r$ = {corr:.3f}) — {title_prefix}')
            row_axes[2].set_xlabel('Predicted STD ($\sigma$)')
            row_axes[2].set_ylabel('Absolute Error ($|\\mu - y|$)')
            row_axes[2].grid(True, alpha=0.2)
            
            # Линия тренда
            m, b = np.polyfit(s_flat, abs_errors, 1)
            x_line = np.linspace(s_flat.min(), s_flat.max(), 100)
            row_axes[2].plot(x_line, m * x_line + b, color='red', linestyle='--', lw=1.5, label='Trend')
            row_axes[2].legend(loc='upper right', fontsize=9)

        plt.tight_layout()
        
        if save_path:
            save_dir = Path(save_path)
            t_path = save_dir / f'Evaluation_board.png'
            plt.savefig(t_path, dpi=600, bbox_inches='tight')
            print(f"📊 График сохранен: {t_path}")
            
        plt.show()

## Переменные для работы блокнота

In [8]:
X_TRAIN = np.load('../processed_data/DataSets/Scenario_3_Train_X.npz')['arr_0']
Y_TRAIN = np.load('../processed_data/DataSets/Scenario_3_Train_y.npz')['arr_0']

X_TRAIN, X_VAL, Y_TRAIN, Y_VAL = train_test_split(X_TRAIN, Y_TRAIN, test_size=0.15, random_state=42)

X_ADD_VAL = np.load('../processed_data/DataSets/Scenario_3_Val_X.npz')['arr_0']
Y_ADD_VAL = np.load('../processed_data/DataSets/Scenario_3_Val_y.npz')['arr_0']

X_TEST = np.load('../processed_data/DataSets/Scenario_3_Test_X.npz')['arr_0']
Y_TEST = np.load('../processed_data/DataSets/Scenario_3_Test_y.npz')['arr_0']

# max_cn = np.array([12, 6, 24, 12, 24, 8, 48])
max_cn = np.array([1, 1, 1, 1, 1, 1, 1])

Y_TRAIN = Y_TRAIN/max_cn
Y_VAL = Y_VAL/max_cn
Y_ADD_VAL = Y_ADD_VAL/max_cn
Y_TEST = Y_TEST/max_cn

In [9]:
E_GRID = np.arange(-10,60.1,0.5)
IN_DIM=141 # Размер входного спектра.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# CRITERION = GaussianNLLLoss()
CRITERION = HybridGaussianMSELoss(1,1,0.5)
DROPOUT = 0.0
NUM_EPOCHS = 100
N_OUT = 1
KERNEL_SIZE = 5

In [10]:
OPTIMIZER = optim.AdamW
OPT_PARAMS = {'lr':1e-3, 'weight_decay': 1e-7}#1e-3}

SCHEDULER = CosineAnnealingWarmRestarts
# SCH_PARAMS = {'T_0':80, 'T_mult':2, 'eta_min':1e-6}
SCH_PARAMS = {'T_0': 100, 'T_mult': 1, 'eta_min': 1e-6}

MODEl_PARAMS = {'in_features':IN_DIM,
                'kernel_size':KERNEL_SIZE,
                'dropout':DROPOUT}

In [11]:
MODEL = XANES_CNN_Stable(in_features=IN_DIM, num_outputs=4, dropout=DROPOUT)
num_params = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
print(f'Число параметров модели = {num_params}')

Число параметров модели = 51720


## Цикл обучения по разным CN

In [ ]:
%%time

save_root =  Path('../processed_data/Results/Scenario3_full/CNN_with_std/')
viz = XANESVisualizer()
spectrum = X_TEST[1]

for i in range(7):
    cn_label = f"CN{i+1}"
    if save_root:
        current_save_path = Path(save_root, cn_label)
        current_save_path.mkdir(parents=True, exist_ok=True)
        
    print(f"\n🚀 Моделирование {cn_label}...")
    model, history, artifacts = fit_cyclic_checkpoints(model_cls = XANES_CNN_Stable,
                                                       model_params = MODEl_PARAMS,
                                                       criterion = CRITERION,
                                                       X_train = X_TRAIN,
                                                       y_train = Y_TRAIN,
                                                       X_val_split = X_VAL,
                                                       y_val_split = Y_VAL,
                                                       X_val_indep = X_ADD_VAL,
                                                       y_val_indep = Y_ADD_VAL,
                                                       optimizer_cls = OPTIMIZER,
                                                       optimizer_params = OPT_PARAMS,
                                                       scheduler_cls = SCHEDULER,
                                                       scheduler_params = SCH_PARAMS,
                                                       n_targets = N_OUT,
                                                       target_index = i,
                                                       X_test = X_TEST,
                                                       y_test = Y_TEST,
                                                       batch_size = 128,
                                                       num_epochs = NUM_EPOCHS,
                                                       save_dir = str(Path(current_save_path, 'checkpoints')),
                                                       device = DEVICE
                                                      )

    plot_metrics(history, current_save_path)
   
    importance_mean = get_probabilistic_feature_importance(model, spectrum, DEVICE, target_idx=None, output_type='mu')
    importance_std = get_probabilistic_feature_importance(model, spectrum, DEVICE, target_idx=None, output_type='std')
    mean_path = Path(current_save_path,'mean')
    mean_path.mkdir(parents=True, exist_ok=True)
    std_path = Path(current_save_path,'std')
    std_path.mkdir(parents=True, exist_ok=True)
    
    viz.plot_feature_importance(spectrum = spectrum, 
                                importance = importance_mean, 
                                title=f"Importance CN{i+1} mean", 
                                save_root=mean_path,
                                energy_grid=E_GRID)
    
    viz.plot_feature_importance(spectrum = spectrum, 
                                importance = importance_std, 
                                title=f"Importance CN{i+1} std", 
                                save_root=std_path,
                                energy_grid=E_GRID)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # Игнорируем предупреждения
        evaluate_and_plot_board(model = model,
                                device = DEVICE,
                                n_out = N_OUT,
                                X_train = X_TRAIN,
                                y_train = Y_TRAIN[:, i].reshape(-1,1),
                                X_test = X_TEST,
                                y_test = Y_TEST[:, i].reshape(-1,1),
                                X_val = X_VAL,
                                y_val = Y_VAL[:, i].reshape(-1,1),
                                X_indep = X_ADD_VAL,
                                y_indep = Y_ADD_VAL[:, i].reshape(-1,1),
                                target_idx = None,
                                save_path = current_save_path
                                )